In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [5]:
df = pd.read_csv(r"C:\Users\lekshmi\Downloads\gold_lstm_features.csv")
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
print(df.shape)
print(df.head())

(2045, 4)
            Gold_Close  USD_Close  avg_sentiment  news_count
Date                                                        
2017-12-18     28663.0      93.69      -0.073070        40.0
2017-12-19     28555.0      93.44      -0.179765        20.0
2017-12-20     28617.0      93.31       0.000650        20.0
2017-12-21     28596.0      93.28       0.050420        20.0
2017-12-22     28757.0      93.35       0.013470        20.0


In [7]:
# Add rolling averages
df['gold_ma7']  = df['Gold_Close'].rolling(7).mean()
df['gold_ma30'] = df['Gold_Close'].rolling(30).mean()
df['usd_ma7']   = df['USD_Close'].rolling(7).mean()

# Drop NaN rows created by rolling
df.dropna(inplace=True)

print("Shape after feature engineering:", df.shape)
print(df.head())

Shape after feature engineering: (2016, 7)
            Gold_Close  USD_Close  avg_sentiment  news_count      gold_ma7  \
Date                                                                         
2018-01-30     30052.0      89.16      -0.065922        40.0  30023.142857   
2018-01-31     30064.0      89.13      -0.023725        20.0  30069.000000   
2018-02-01     30512.0      88.67      -0.070993        40.0  30169.857143   
2018-02-02     30399.0      89.19      -0.070993        40.0  30244.571429   
2018-02-05     30286.0      89.55      -0.017390        20.0  30250.571429   

               gold_ma30    usd_ma7  
Date                                 
2018-01-30  29426.133333  89.737143  
2018-01-31  29472.833333  89.531429  
2018-02-01  29538.066667  89.284286  
2018-02-02  29597.466667  89.151429  
2018-02-05  29653.800000  89.200000  


In [8]:
from sklearn.preprocessing import MinMaxScaler

features = ['Gold_Close', 'USD_Close', 'avg_sentiment', 'news_count',
            'gold_ma7', 'gold_ma30', 'usd_ma7']

scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[features])

print("Scaled shape:", scaled.shape)
print("Min values:", scaled.min(axis=0).round(4))
print("Max values:", scaled.max(axis=0).round(4))

Scaled shape: (2016, 7)
Min values: [0. 0. 0. 0. 0. 0. 0.]
Max values: [1. 1. 1. 1. 1. 1. 1.]


In [9]:
LOOKBACK = 60
FORECAST = 30

X, y = [], []
for i in range(LOOKBACK, len(scaled) - FORECAST):
    X.append(scaled[i-LOOKBACK:i])       # 60 days input
    y.append(scaled[i:i+FORECAST, 0])    # next 30 days gold price

X, y = np.array(X), np.array(y)
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1926, 60, 7)
y shape: (1926, 30)


In [10]:
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1540, 60, 7)
X_test: (386, 60, 7)
y_train: (1540, 30)
y_test: (386, 30)


In [11]:
import numpy as np

np.save(r"C:\Users\lekshmi\Downloads\X_train.npy", X_train)
np.save(r"C:\Users\lekshmi\Downloads\X_test.npy", X_test)
np.save(r"C:\Users\lekshmi\Downloads\y_train.npy", y_train)
np.save(r"C:\Users\lekshmi\Downloads\y_test.npy", y_test)

# Also save the scaler for inverse transforming predictions later
import pickle
with open(r"C:\Users\lekshmi\Downloads\scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("✅ Saved:")
print("  X_train.npy —", X_train.shape)
print("  X_test.npy  —", X_test.shape)
print("  y_train.npy —", y_train.shape)
print("  y_test.npy  —", y_test.shape)
print("  scaler.pkl  — for inverse scaling predictions")

✅ Saved:
  X_train.npy — (1540, 60, 7)
  X_test.npy  — (386, 60, 7)
  y_train.npy — (1540, 30)
  y_test.npy  — (386, 30)
  scaler.pkl  — for inverse scaling predictions
